# elastic_transport smoke test

Tests basic API surface of `elastic_transport` — the low-level HTTP transport library shared by Elasticsearch and Elastic clients.

In [ ]:
# Example 1: Public API surface — verify core classes are exposed.
from elastic_transport import (
    Transport,
    NodeConfig,
    Urllib3HttpNode,
    ApiResponse,
    ConnectionError,
    TransportError,
)

for cls in (Transport, NodeConfig, Urllib3HttpNode, ApiResponse, ConnectionError, TransportError):
    print(f"{cls.__name__}: {cls.__module__}")

assert issubclass(ConnectionError, TransportError)

In [ ]:
# Example 3: Exception hierarchy — exercise transport error types.
# elastic-transport 9.x hierarchy: ConnectionTimeout subclasses TransportError DIRECTLY
# (no longer a ConnectionError child — timeouts are retryable, connection failures aren't),
# and TlsError is a ConnectionError subclass.
from elastic_transport import (
    TransportError,
    ConnectionError,
    ConnectionTimeout,
    TlsError,
    SerializationError,
    ApiError,
)

assert issubclass(ConnectionError, TransportError)
assert issubclass(ConnectionTimeout, TransportError)
assert not issubclass(ConnectionTimeout, ConnectionError)
assert issubclass(TlsError, ConnectionError)
assert issubclass(TlsError, TransportError)
assert issubclass(SerializationError, TransportError)
assert issubclass(ApiError, Exception)
assert not issubclass(ApiError, TransportError)

err = ConnectionError("refused")
print(f"err = {err!r}")
print(f"message = {err.message}")

In [ ]:
# Example 5: Transport construction with a single NodeConfig (no network call).
from elastic_transport import Transport, NodeConfig, Urllib3HttpNode

node = NodeConfig(scheme="http", host="localhost", port=9200)
transport = Transport(
    [node],
    node_class=Urllib3HttpNode,
)

print(f"transport = {type(transport).__name__}")
print(f"node_pool size = {len(list(transport.node_pool.all()))}")

nodes = list(transport.node_pool.all())
assert len(nodes) == 1
assert nodes[0].config.host == "localhost"
assert nodes[0].config.port == 9200
print("OK elastic_transport")